# `source_final.ipynb` — finalized pipeline (incremental)

This notebook is the **production** version of the sources pipeline.

Rule: we only copy stages into here once they were validated in `sources_test.ipynb`.

**Status**
- ✅ Stage B (Chapter Blueprint) finalized: `coverage_v1` (wins vs baseline)
- ✅ Stage C scoring finalized: `w_embed_max=0.0`, `w_embed=0.7`, `cite_weight=0.08`
- ✅ Stage C.3 finalized: LLM rerank **within Stage C top-50** + deterministic tie-break (model: `gpt-5-nano`)
- ✅ Stage D decision: not used by default (diversity modes hurt relevance in LOCO)
- ⏳ Stage A/C.2 plumbing (fetch + embeddings + TF-IDF): still to copy into this notebook


In [1]:
from __future__ import annotations

import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field

# OpenAI / Agents SDK (used for schema-validated JSON output)
from openai import OpenAI
from agents import Agent, Runner, ModelSettings


# -----------------------------
# Minimal .env loader (notebook convenience)
# -----------------------------
def load_dotenv_minimal(path: str = ".env") -> None:
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and (k not in os.environ):
            os.environ[k] = v


load_dotenv_minimal(".env")

if not os.getenv("OPENAI_API_KEY", "").strip():
    raise RuntimeError("Missing OPENAI_API_KEY. Add it to your environment or .env file.")

client = OpenAI()  # validates credentials on first real request

# -----------------------------
# Output dirs (cache)
# -----------------------------
OUT_DIR = Path("final_pipeline")
STAGEB_DIR = OUT_DIR / "stageB_blueprints"
STAGEB_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Finalized decision from Stage B A/B testing
# -----------------------------
STAGEB_FINAL_VARIANT = "coverage_v1"  # ✅ finalized winner

# Model pricing (USD per 1M tokens) — keep updated if pricing changes.
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
    "text-embedding-3-small": {"input": 0.02, "cached": 0.0, "output": 0.0},
}

# Stage B model (only a few calls per chapter, but we still cost-track)
BLUEPRINT_MODEL = "gpt-5-mini"

# Rebuild controls
FORCE_REBUILD_BLUEPRINTS = False

# -----------------------------
# Chapter specs (edit/replace these to use the pipeline on new chapters)
# -----------------------------
CHAPTERS: List[Dict[str, Any]] = [
    {
        "chapter_id": "platform_theory",
        "title": "Theoretische Fundierung: Mechanismen der Plattformökonomie",
        "original_text_language": "de",
        "original_text": (
            "Aufarbeitung der Begriffe und Modelle, die zur Analyse von Plattformwachstum und Wettbewerb benötigt werden. "
            "(2.1) Two-/Multi-Sided Markets: Rolle mehrerer Nutzergruppen, Wertschaffung durch Vermittlung. "
            "(2.2) Direkte und indirekte Netzwerkexternalitäten, jeweils positiv und negativ, und ihre Bedeutung für Nutzen, Qualität und Skalierung. "
            "(2.3) Kritische Masse und Wachstumspfad inklusive Henne-Ei-Dilemma. "
            "(2.4) Multi-Homing und Switching Costs als Treiber oder Bremse von Bindung, Marktteilung und Wechselverhalten. "
            "(2.5) Plattform-Governance über Regeln, Rankings und Zugang als Instrument zur Qualitätssteuerung und Stabilisierung der Interaktionen. "
            "(2.6) Monetarisierung und Preisstruktur, einschließlich Gebühren/Provisionen, und deren Wechselwirkung mit Netzwerkeffekten. "
            "(2.7) Wettbewerbsdynamik: Bedingungen, unter denen Netzwerkeffekte \u201eWinner-takes-most\u201c begünstigen versus Koexistenz mehrerer Plattformen. "
            "Vertiefung erfolgt nur in diesen Konzepten; keine zusätzlichen Markt- oder Organisationstheorien."
        ),
    },
    {
        "chapter_id": "platform_methodology",
        "title": "Methodik: Strukturierte Literaturanalyse und Entwicklung des Frameworks",
        "original_text_language": "de",
        "original_text": (
            "Festlegung eines nachvollziehbaren Vorgehens zur Herleitung und späteren Anwendung eines kompakten Analyse-Frameworks. "
            "(3.1) Strukturierte Literaturanalyse: Definition klarer Suchbegriffe entlang der Themen Two-/Multi-Sided Markets, direkte/indirekte (positive/negative) Netzwerkeffekte, Multi-Homing, Plattform-Governance (Regeln, Rankings, Zugang), Switching Costs, Monetarisierung (Gebühren/Provisionen), Preisstruktur, Wettbewerb und \u201eWinner-takes-most\u201c. "
            "Festlegung transparenter Auswahl- und Ausschlusskriterien mit Fokus auf Passung zu diesen Mechanismen und Verwendbarkeit für ein einheitliches Analysegerüst. "
            "(3.2) Framework-Entwicklung: Verdichtung der Literatur zu den Dimensionen Netzwerkeffekte, Multi-Homing, Governance, Preisstruktur, Wettbewerb. "
            "(3.3) Operationalisierung für die Fallanalyse: Festlegung, wie die Dimensionen mit öffentlich verfügbaren Daten/Dokumenten diskutiert werden (Nutzer-/Anbieterwachstum, Gebührenänderungen, Regeländerungen, Marktanteile, Qualitätsindikatoren). "
            "Auswahl eines konkreten Plattformfalls aus Airbnb, Uber, eBay oder einem App-Store entlang der Verfügbarkeit solcher Informationen."
        ),
    },
    {
        "chapter_id": "platform_empirical_case",
        "title": "Empirische Anwendung: Analyse eines konkreten Plattformfalls mit dem Framework",
        "original_text_language": "de",
        "original_text": (
            "Darstellung und Analyse des ausgewählten Plattformfalls (Airbnb, Uber, eBay oder ein App-Store) anhand öffentlich verfügbarer Daten und Dokumente. "
            "(4.1) Kurzprofil der Plattform und der relevanten Nutzergruppen sowie Abgrenzung des betrachteten Marktbereichs, soweit für die Framework-Anwendung notwendig. "
            "(4.2) Anwendung des Frameworks entlang der Dimensionen: Identifikation und Einordnung direkter und indirekter Netzwerkeffekte (positiv/negativ) und deren Zusammenhang mit Nutzer-/Anbieterwachstum und dem Erreichen bzw. Überschreiten kritischer Masse; Beobachtung von Multi-Homing und möglichen Wechselbarrieren (Switching Costs); "
            "Analyse von Governance-Änderungen (Regeln, Rankings, Zugang) und deren Bezug zu Qualitätsindikatoren; Einordnung der Preisstruktur und Monetarisierung über Gebühren/Provisionen inklusive relevanter Gebührenänderungen; Diskussion der Wettbewerbsentwicklung anhand verfügbarer Marktanteilsindikatoren. "
            "(4.3) Fallbezogene Synthese, welche Netzwerkeffekte im Fall dominieren und welche Konsequenzen sich für Strategie und ggf. Regulierung ableiten lassen. "
            "Keine neue Theorieentwicklung über das Framework hinaus."
        ),
    },
]

print("Config OK")
print("Stage B finalized variant:", STAGEB_FINAL_VARIANT)
print("Chapters:", [c["chapter_id"] for c in CHAPTERS])


Config OK
Stage B finalized variant: coverage_v1
Chapters: ['platform_theory', 'platform_methodology', 'platform_empirical_case']


## Stage B (finalized): Chapter Blueprint generation (`coverage_v1`)

This is the only Stage B variant that goes into `source_final.ipynb`.


In [2]:
# -----------------------------
# Stage B: build blueprints (cached)
# -----------------------------

class ChapterBlueprint(BaseModel):
    chapter_id: str
    language: str = Field("en")

    scope_statement: str
    must_cover: List[str]
    should_cover: List[str]
    must_avoid: List[str]

    main_query: str
    facet_queries: List[str]
    keywords: List[str]
    key_concepts: List[str]

    preferred_source_types: Optional[List[str]] = None
    negative_query_terms: Optional[List[str]] = None

    scoring_guidance: str
    notes: Optional[str] = None


def price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})


def cost_from_usage(usage, model: str) -> dict:
    """Token totals + estimated cost using MODEL_PRICES_USD_PER_1M."""

    prices = price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    # Fallback: aggregated totals
    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }


BASE_BLUEPRINT_INSTRUCTIONS = (
    "You create a chapter blueprint (rubric) and search queries for academic literature retrieval.\n"
    "Return ONLY the structured output fields (no extra text).\n\n"
    "Constraints:\n"
    "- language must be 'en'\n"
    "- scope_statement: 1 sentence, <= 30 words\n"
    "- must_cover: 4\u20138 bullets, each <= 16 words\n"
    "- should_cover: 3\u20138 bullets, each <= 16 words\n"
    "- must_avoid: 3\u20138 bullets, each <= 16 words\n"
    "- main_query: <= 18 words\n"
    "- facet_queries: 8\u201314 items, each <= 14 words\n"
    "- keywords: 20\u201345 items\n"
    "- key_concepts: 10\u201322 items\n"
    "- preferred_source_types: 2\u20136 items\n"
    "- negative_query_terms: 0\u201312 items derived from must_avoid (soft negatives)\n"
    "- scoring_guidance: <= 80 words\n"
    "- Do NOT contradict yourself: if something is in must_avoid, do not emphasize it in keywords/facets.\n"
    "- Do NOT hardcode any domain. Follow the given chapter spec.\n"
)

COVERAGE_V1_INSTRUCTIONS = BASE_BLUEPRINT_INSTRUCTIONS + (
    "\nAdditional requirements (coverage_v1):\n"
    "- facet_queries must be semantically diverse (avoid near-duplicates).\n"
    "- Ensure each must_cover bullet is explicitly targeted by at least one facet_query.\n"
)

BLUEPRINT_INSTRUCTIONS = COVERAGE_V1_INSTRUCTIONS

blueprint_agent = Agent(
    name="Chapter Blueprint Builder (coverage_v1)",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=BLUEPRINT_INSTRUCTIONS,
    output_type=ChapterBlueprint,
)


async def get_or_create_blueprint(chapter: dict) -> tuple[dict, dict]:
    out_path = STAGEB_DIR / f"{chapter['chapter_id']}.json"

    if (not FORCE_REBUILD_BLUEPRINTS) and out_path.exists():
        bp = json.loads(out_path.read_text(encoding="utf-8"))
        return bp, {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}

    prompt = (
        "Create a ChapterBlueprint for academic literature retrieval.\n"
        "Return ONLY the structured output fields required by the schema.\n\n"
        "CHAPTER_SPEC_JSON:\n"
        + json.dumps(chapter, ensure_ascii=False, indent=2)
    )

    res = await Runner.run(blueprint_agent, prompt)
    bp = res.final_output.model_dump()
    bp["_meta"] = {
        "blueprint_variant": STAGEB_FINAL_VARIANT,
        "model": BLUEPRINT_MODEL,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

    usage = res.context_wrapper.usage
    return bp, cost_from_usage(usage, model=BLUEPRINT_MODEL)


blueprints: Dict[str, dict] = {}
bp_totals = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
for ch in CHAPTERS:
    bp, u = await get_or_create_blueprint(ch)
    blueprints[ch["chapter_id"]] = bp
    for k in bp_totals:
        bp_totals[k] += u.get(k, 0)

print("Stage B blueprints ready")
print("Stage B cost summary:", bp_totals)
for cid, bp in blueprints.items():
    print("-", cid, "| facets:", len(bp.get("facet_queries", [])), "| main:", bp.get("main_query"))


Stage B blueprints ready
Stage B cost summary: {'requests': 3, 'input_tokens': 2618, 'cached_input_tokens': 0, 'output_tokens': 9278, 'cost_usd': 0.0192105}
- platform_theory | facets: 14 | main: Platform mechanisms: multi-sided markets, network effects, multi-homing, governance, pricing, competition
- platform_methodology | facets: 12 | main: two-sided markets AND network effects AND governance AND pricing AND multi-homing AND case study operationalization
- platform_empirical_case | facets: 12 | main: empirical analysis digital platform case study public data network effects governance fees


## Stage C (finalized): scoring weights

Chosen via `sources_test.ipynb` Stage C grid search (`stageC_grid_v1_best`).

**Final weights**
- `w_embed_max = 0.0` (use `score_embed_mean_top3` only)
- `w_embed = 0.7`
- `cite_weight = 0.08`


In [ ]:
import numpy as np
import pandas as pd

# Finalized Stage C hyperparams (grid-search winner)
STAGEC_FINAL_RUN = "20260130_184909_7370e7a6e85f"  # reference from eval_dataset/experiments
STAGEC_W_EMBED_MAX = 0.0
STAGEC_W_EMBED = 0.7
STAGEC_CITE_WEIGHT = 0.08


def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out


def add_stagec_final_scores(df_in: pd.DataFrame) -> pd.DataFrame:
    """Adds `score_stageC_final` based on finalized Stage C weights.

    Required columns:
    - chapter_id
    - score_embed_max
    - score_embed_mean_top3
    - score_tfidf
    - score_cite_norm
    """
    required = [
        "chapter_id",
        "score_embed_max",
        "score_embed_mean_top3",
        "score_tfidf",
        "score_cite_norm",
    ]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage C scoring: {missing}")

    df = df_in.copy()

    emb_max = pd.to_numeric(df["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(df["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    df["_emb_raw"] = float(STAGEC_W_EMBED_MAX) * emb_max + (1.0 - float(STAGEC_W_EMBED_MAX)) * emb_b

    df["_emb_n"] = minmax_by_group(df, "_emb_raw")
    df["_tf_n"] = minmax_by_group(df, "score_tfidf")
    df["_cite_n"] = minmax_by_group(df, "score_cite_norm")

    base = float(STAGEC_W_EMBED) * df["_emb_n"] + (1.0 - float(STAGEC_W_EMBED)) * df["_tf_n"]
    df["score_stageC_final"] = (1.0 - float(STAGEC_CITE_WEIGHT)) * base + float(STAGEC_CITE_WEIGHT) * df["_cite_n"]
    return df


print("Stage C finalized weights:")
print("- STAGEC_W_EMBED_MAX:", STAGEC_W_EMBED_MAX)
print("- STAGEC_W_EMBED:", STAGEC_W_EMBED)
print("- STAGEC_CITE_WEIGHT:", STAGEC_CITE_WEIGHT)


In [ ]:
# -----------------------------
# Stage C.3 (finalized): LLM rerank within Stage C top-N
# -----------------------------
#
# Final decision from LOCO testing (2026-01-31):
# - Use the LLM only as a shortlist reranker to avoid full-rewrite errors.
# - Select top-N by `score_stageC_final`, then order that shortlist by LLM score.

import asyncio
import time
import random
from typing import Dict, Tuple

import numpy as np
import pandas as pd

STAGEC3_TOPN = 50
STAGEC3_MODEL = "gpt-5-nano"
STAGEC3_PROMPT_VERSION = "v1"

STAGEC3_CONCURRENCY = 8
STAGEC3_MAX_RETRIES = 8
STAGEC3_BACKOFF_INITIAL = 1.0
STAGEC3_BACKOFF_MAX = 30.0
STAGEC3_ABSTRACT_MAX_CHARS = 2000

STAGEC3_FORCE_RERANK = False

STAGEC3_DIR = OUT_DIR / "stageC3_rerank_cache_v1"
STAGEC3_DIR.mkdir(parents=True, exist_ok=True)


class StageC3RerankOut(BaseModel):
    score: int = Field(..., ge=0, le=100)
    notes: str = Field("", max_length=140)


stagec3_agent = Agent(
    name=f"StageC3 Shortlist Rerank ({STAGEC3_PROMPT_VERSION})",
    model=STAGEC3_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You score academic papers for inclusion in a specific thesis chapter. "
        "Use the rubric strictly. Return ONLY the structured output."
    ),
    output_type=StageC3RerankOut,
)


def _stagec3_rubric_signature(bp: ChapterBlueprint) -> str:
    payload = {
        "scope_statement": bp.scope_statement,
        "must_cover": bp.must_cover,
        "must_avoid": bp.must_avoid,
        "scoring_guidance": bp.scoring_guidance,
        "prompt_version": STAGEC3_PROMPT_VERSION,
        "model": STAGEC3_MODEL,
        "abstract_max_chars": STAGEC3_ABSTRACT_MAX_CHARS,
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:8]


def _stagec3_prompt(bp: ChapterBlueprint, title: str, abstract: str) -> str:
    scope = bp.scope_statement
    must_cover = bp.must_cover or []
    must_avoid = bp.must_avoid or []
    guidance = bp.scoring_guidance or ""

    abs_txt = (abstract or "").strip()
    if STAGEC3_ABSTRACT_MAX_CHARS and len(abs_txt) > int(STAGEC3_ABSTRACT_MAX_CHARS):
        abs_txt = abs_txt[: int(STAGEC3_ABSTRACT_MAX_CHARS)]

    lines = []
    lines.append("Score this paper for inclusion in the chapter.")
    lines.append("Return only the schema fields.")
    lines.append("")
    lines.append("RUBRIC")
    lines.append(f"SCOPE: {scope}")
    if guidance:
        lines.append(f"GUIDANCE: {guidance}")
    if must_cover:
        lines.append("MUST_COVER:")
        for b in must_cover:
            lines.append(f"- {b}")
    if must_avoid:
        lines.append("MUST_AVOID:")
        for b in must_avoid:
            lines.append(f"- {b}")
    lines.append("")
    lines.append("PAPER")
    lines.append(f"TITLE: {(title or '').strip()}")
    if abs_txt:
        lines.append(f"ABSTRACT: {abs_txt}")
    else:
        lines.append("ABSTRACT: (missing)")
    lines.append("")
    return "\n".join(lines)


def _minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


async def stagec3_rerank_topn(
    df_in: pd.DataFrame,
    blueprints_by_chapter_id: Dict[str, ChapterBlueprint],
    *,
    topn: int = STAGEC3_TOPN,
    id_col: str = "merge_key",
) -> Tuple[pd.DataFrame, dict]:
    """Adds Stage C.3 shortlist rerank scores.

    Required columns in df_in:
    - chapter_id
    - title
    - abstract
    - score_stageC_final

    Returns (df_out, totals).
    """
    required = ["chapter_id", "title", "abstract", "score_stageC_final"]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage C.3: {missing}")

    df = df_in.copy()
    if id_col not in df.columns:
        years = df["year"] if "year" in df.columns else pd.Series([""] * len(df), index=df.index)
        df[id_col] = [
            hashlib.sha1(f"{t}|{y}".encode("utf-8")).hexdigest()[:16]
            for t, y in zip(df["title"].fillna("").astype(str), years.fillna("").astype(str))
        ]

    df["score_llm_rerank_v1"] = np.nan
    df["llm_notes"] = ""

    sem = asyncio.Semaphore(int(STAGEC3_CONCURRENCY))
    t0 = time.time()

    totals = {
        "seconds": 0.0,
        "requests": 0,
        "input_tokens": 0,
        "cached_input_tokens": 0,
        "output_tokens": 0,
        "cost_usd": 0.0,
        "cached_files": 0,
    }

    async def _run_one(chapter_id: str, doc_id: str, title: str, abstract: str) -> dict:
        bp = blueprints_by_chapter_id[chapter_id]
        sig = _stagec3_rubric_signature(bp)
        cache_dir = STAGEC3_DIR / sig / chapter_id
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path = cache_dir / f"{doc_id}.json"

        if cache_path.exists() and not STAGEC3_FORCE_RERANK:
            payload = json.loads(cache_path.read_text(encoding="utf-8"))
            payload["_meta"] = payload.get("_meta") or {}
            payload["_meta"]["llm_cached"] = True
            return payload

        prompt = _stagec3_prompt(bp, title=title, abstract=abstract)
        last_err = None
        for attempt in range(int(STAGEC3_MAX_RETRIES)):
            try:
                async with sem:
                    res = await Runner.run(stagec3_agent, prompt)
                usage = getattr(res, "usage", None)
                meta = cost_from_usage(usage, model=STAGEC3_MODEL) if usage is not None else {"requests": 1, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
                out = {
                    "score": int(res.final_output.score),
                    "notes": str(res.final_output.notes or ""),
                    "_meta": {**meta, "llm_cached": False, "rubric_sig": sig, "model": STAGEC3_MODEL},
                }
                cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
                return out
            except Exception as e:
                last_err = e
                backoff = min(float(STAGEC3_BACKOFF_MAX), float(STAGEC3_BACKOFF_INITIAL) * (2 ** attempt))
                backoff = backoff * (0.75 + 0.5 * random.random())
                await asyncio.sleep(backoff)

        raise RuntimeError(f"Stage C.3 rerank failed after retries for chapter_id={chapter_id} doc_id={doc_id}: {last_err}")

    tasks = []
    shortlist_idx = []
    for chapter_id, g in df.groupby("chapter_id"):
        chapter_id = str(chapter_id)
        if chapter_id not in blueprints_by_chapter_id:
            raise KeyError(f"Missing blueprint for chapter_id '{chapter_id}'")
        sort_cols = ["score_stageC_final"]
        asc = [False]
        if id_col in g.columns:
            sort_cols.append(id_col)
            asc.append(True)
        idx = g.sort_values(sort_cols, ascending=asc, kind="mergesort").head(int(topn)).index
        shortlist_idx.extend(idx.tolist())
        for i in idx:
            row = df.loc[i]
            tasks.append(_run_one(chapter_id, str(row[id_col]), str(row.get("title") or ""), str(row.get("abstract") or "")))

    results = await asyncio.gather(*tasks)

    for i, payload in zip(shortlist_idx, results):
        df.loc[i, "score_llm_rerank_v1"] = payload.get("score")
        df.loc[i, "llm_notes"] = payload.get("notes", "")
        m = payload.get("_meta") or {}
        totals["requests"] += int(m.get("requests", 1) or 1)
        totals["input_tokens"] += int(m.get("input_tokens", 0) or 0)
        totals["cached_input_tokens"] += int(m.get("cached_input_tokens", 0) or 0)
        totals["output_tokens"] += int(m.get("output_tokens", 0) or 0)
        totals["cost_usd"] += float(m.get("cost_usd", 0.0) or 0.0)
        totals["cached_files"] += int(bool(m.get("llm_cached", False)))

    totals["seconds"] = float(time.time() - t0)
    print("Stage C.3 rerank totals:", totals)

    df["_in_topn"] = False
    df.loc[shortlist_idx, "_in_topn"] = True

    # Final Stage C.3 score: topN reordered by LLM with deterministic tie-breaks;
    # tail stays Stage C.
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    for _, g in df[df["_in_topn"]].groupby("chapter_id"):
        llm_n = _minmax_series(g["score_llm_rerank_v1"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(g["score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(g["score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        top = pd.DataFrame({
            "ix": g.index.to_list(),
            "doc_id": df.loc[g.index, id_col].fillna("").astype(str).to_list(),
            "llm_n": llm_n,
            "stagec": stagec,
            "cite": cite,
        })
        top = top.sort_values(["llm_n", "stagec", "cite", "doc_id"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_final"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    return df, totals


print("Stage C.3 finalized params:")
print("- STAGEC3_TOPN:", STAGEC3_TOPN)
print("- STAGEC3_MODEL:", STAGEC3_MODEL)


## Next stages (not yet finalized)

The following stages will be appended here once they are validated:
- Stage A: API retrieval (OpenAlex + Semantic Scholar) + merge/dedupe
- Stage C.2: embeddings + TF-IDF scoring (to produce Stage C inputs)
- (Optional) Stage D: diversity-mode selection (disabled by default)
